In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, sum, lit, current_timestamp, date_format, when, corr, lag, min, max, round
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

######## Carrega a tabela Silver ########
df_silver = spark.table("grao_direto.default.tabela_silver_grain_logistic")

def salvar_tabela(df, nome_tabela):
    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"grao_direto.default.{nome_tabela}")

# 1. Desempenho e Satisfação
######## Desempenho por Método de Envio ########
agg_metodo_envio = df_silver.groupBy("metodo_de_envio") \
    .agg(
        count("id_envio").alias("total_envios"),
        round(avg("avaliacao_do_cliente"),2).alias("media_avaliacao_cliente"),
        round(avg("tempo_entrega_dias"),2).alias("media_dias_entrega"),
        sum(when(col("chegou_no_tempo") == 'Sim', 0).otherwise(1)).alias("total_atrasos")
    ) \
    .withColumn("percentual_atrasos", round(((col("total_atrasos") / col("total_envios")) * 100),2))


######## Satisfação por Destino ########
agg_satisfacao_destino = df_silver.groupBy("destino") \
    .agg(
        count("id_envio").alias("total_envios"),
        round(avg("avaliacao_do_cliente"),2).alias("media_avaliacao_cliente"),
        round(avg("avaliacao_da_entrega"),2).alias("media_avaliacao_entrega")
    ) \
    .orderBy(col("media_avaliacao_cliente").desc())

######## Relação entre Satisfação e Desempenho Operacional ########
agg_satisfacao = df_silver.groupBy("chegou_no_tempo", "importancia") \
    .agg(
        count("id_envio").alias("total_envios"),
        round(avg("avaliacao_do_cliente"),2).alias("media_avaliacao_cliente"),
        round(avg("avaliacao_da_entrega"),2).alias("media_avaliacao_entrega"),
        round(avg("ligacoes_do_cliente"),2).alias("media_ligacoes_cliente"),
        round(avg("tempo_entrega_dias"),2).alias("media_tempo_entrega")
    ) \
    .orderBy(col("chegou_no_tempo"), col("importancia"))


salvar_tabela(agg_metodo_envio, "gold_desempenho_por_metodo_envio")
salvar_tabela(agg_satisfacao_destino, "gold_satisfacao_por_destino")
salvar_tabela(agg_satisfacao, "gold_satisfacao_vs_desempenho")


# 2. Temporal / Evolução
######## Análise Mensal de Envios ########
agg_envios_mensal = df_silver.withColumn("mes_envio", date_format(col("data_do_envio"), "yyyy-MM")) \
    .groupBy("mes_envio") \
    .agg(
        count("id_envio").alias("total_envios"),
        round(sum("preco"),2).alias("receita_total"),
        round(sum("peso_em_gramas"),2).alias("peso_total_gramas"),
        round(avg("desconto"),2).alias("media_desconto_aplicado")
    ) \
    .orderBy("mes_envio")

######## Produtividade Mensal com Comparativo ########
agg_mensal_comparativo = df_silver.withColumn("mes_envio", date_format(col("data_do_envio"), "yyyy-MM")) \
    .groupBy("mes_envio") \
    .agg(
        count("id_envio").alias("total_envios"),
        round(sum("preco"),2).alias("receita_total"),
        round(avg("preco"),2).alias("ticket_medio"),
        round(sum("peso_em_gramas"),2).alias("peso_total_gramas")
    ) \
    .withColumn("mes_anterior", lag("total_envios").over(Window.orderBy("mes_envio"))) \
    .withColumn("variacao_envios", 
                ((col("total_envios") - col("mes_anterior")) / col("mes_anterior") * 100)) \
    .orderBy("mes_envio")


salvar_tabela(agg_envios_mensal, "gold_analise_mensal_envios")
salvar_tabela(agg_mensal_comparativo, "gold_produtividade_mensal_comparativo")


# 3. Logística Operacional
######## Eficiência Logística por Método e Destino ########
agg_eficiencia = df_silver.groupBy("metodo_de_envio", "destino") \
    .agg(
        count("id_envio").alias("total_envios"),
        round(avg("tempo_entrega_dias"),2).alias("media_tempo_entrega"),
        min("tempo_entrega_dias").alias("min_tempo_entrega"),
        max("tempo_entrega_dias").alias("max_tempo_entrega"),
        round((sum(when(col("chegou_no_tempo") == "Sim", 1).otherwise(0)) / count("*") * 100),2).alias("taxa_entrega_ontempo"),
        round(avg("peso_em_gramas"),2).alias("peso_medio_gramas")
    ) \
    .orderBy(col("media_tempo_entrega"))

######## Desempenho por Corredor de Armazenagem ########
agg_corredor = df_silver.groupBy("corredor_de_armazenagem") \
    .agg(
        count("id_envio").alias("total_envios"),
        round(avg("tempo_entrega_dias"),2).alias("media_tempo_entrega"),
        round(avg("avaliacao_do_cliente"),2).alias("media_avaliacao_cliente"),
        round(avg("avaliacao_da_entrega"),2).alias("media_avaliacao_entrega"),
        round((sum(when(col("chegou_no_tempo") == "Não", 1).otherwise(0)) / count("*") * 100),2).alias("taxa_atraso_percent"),
        round(sum("peso_em_gramas"),2).alias("peso_total_gramas")
    ) \
    .orderBy(col("taxa_atraso_percent").desc())

salvar_tabela(agg_eficiencia, "gold_eficiencia_logistica")
salvar_tabela(agg_corredor, "gold_desempenho_por_corredor")


# 4. Financeiro / Rentabilidade
######## Relação entre Peso e Custo de Envio ########
agg_peso_custo = df_silver.withColumn("peso_kg", col("peso_em_gramas") / 1000) \
    .withColumn("custo_por_kg", col("preco") / col("peso_kg")) \
    .groupBy("metodo_de_envio") \
    .agg(
        round( avg("peso_kg"),2).alias("peso_medio_kg"),
        round(avg("preco"),2).alias("custo_medio_envio"),
        round( avg("custo_por_kg"),2).alias("custo_medio_por_kg"),
        corr("peso_kg", "preco").alias("correlacao_peso_custo")
    )

######## Rentabilidade por Importância do Cliente ########
agg_rentabilidade = df_silver.groupBy("importancia") \
    .agg(
        count("id_envio").alias("total_envios"),
        sum("preco").alias("receita_total"),
        round(avg("desconto"),2).alias("desconto_medio"),
        round((sum("desconto") / sum("preco") * 100),2).alias("percentual_desconto"),
        round(avg("qtd_itens"),2).alias("media_itens_por_envio"),
       round( avg("preco"),2).alias("ticket_medio")
    ) \
    .orderBy(col("receita_total").desc())

salvar_tabela(agg_peso_custo, "gold_relacao_peso_custo")
salvar_tabela(agg_rentabilidade, "gold_rentabilidade_por_importancia") 

print("Camada Gold concluída com sucesso!")